## 35. 프로젝트 루트 설정

In [2]:
from pathlib import Path
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\dev\ai-data-analysis
데이터 폴더: c:\dev\ai-data-analysis\data\raw
데이터 폴더 존재: True


## 36. pandas와 CSV 불러오기

In [3]:
import pandas as pd
customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

In [4]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}
for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [5]:
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}

for name, series in key_checks.items():
    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


## 38. Series와 DataFrame 선택

In [6]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]
print(type(city_series))
print(type(customer_view))
display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


## 39. 단일 조건 필터링

In [7]:
customers_over_30 = customers[
    customers["age"] >= 30
]
print(len(customers), len(customers_over_30))
display(customers_over_30.head())

150 111


,customer_id,name,gender,age,city,signup_date
1,2,김정호,F,32,대구,2025-12-04
2,3,이경수,F,61,성남,2024-07-14
3,4,조영호,F,55,울산,2026-05-15
5,6,김지원,F,32,성남,2026-07-29
6,7,이상현,F,53,인천,2025-01-13


## 40. 복합 조건 필터링

In [8]:
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]
display(seoul_over_30.head())

,customer_id,name,gender,age,city,signup_date
8,9,송지민,M,69,서울,2025-11-20
14,15,장정식,M,69,서울,2026-07-06
29,30,이민재,F,32,서울,2023-08-15
47,48,김예은,F,47,서울,2025-05-03
65,66,김재호,F,39,서울,2026-01-04


서울 또는 부산

In [9]:
seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)


city
부산    16
서울    15
Name: count, dtype: int64

완료 주문이 아닌 주문:

In [10]:
not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)

order_status
cancelled    64
refunded     52
Name: count, dtype: int64

## 41. 상품 가격 정렬

In [11]:
expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head(10)
)
display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


## 42. 작업용 복사본과 파생 컬럼

In [12]:
order_items_work = order_items.copy()

# 데이터 플레임에 새로운 컬럼을 추가할 때는 기존 데이터 프레임을 수정
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

In [13]:
print(order_items_work.head())

   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000


## 43. 수작업 검증

 
[출처] Chapter 04. pandas로 데이터에 질문하기|작성자 아토믹데브

In [14]:
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True


## 44. 전체 주문상세 금액

In [15]:
all_order_amount = order_items_work["line_total"].sum()

print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 255610000


## 45. 병합용 주문 컬럼 선택

In [16]:
orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()

## 46. 주문상세와 주문 병합

In [19]:
order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)

## 47. 병합 검증

In [20]:
print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

병합 전 행 수: 764
병합 후 행 수: 764


order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64

정상 기준:

• 행 수 유지
• order_match가 모두 both

미매칭 확인:

In [21]:
unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match


## 48. 완료 주문 분석셋

In [22]:
display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

order_status
completed    474
cancelled    162
refunded     128
Name: count, dtype: int64

In [23]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [24]:
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)
print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)
print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)

완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


## 49. 필요한 상품 정보만 선택

In [27]:
products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()

print(products_for_merge.head())

   product_id product_name category
0           1  전자기기 상품 001     전자기기
1           2    도서 상품 002       도서
2           3  전자기기 상품 003     전자기기
3           4  생활용품 상품 004     생활용품
4           5    식품 상품 005       식품


## 50. 완료 주문상세와 상품 병합

In [28]:
completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [29]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 51. 카테고리별 매출

In [30]:
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


## 52. 카테고리 합계 검증

In [31]:
category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

148990000
148990000
True


합계가 다르면 카테고리 결측, 상품 미매칭, 중복 병합과 필터 범위 차이를 확인합니다.

## 53. 상품별 매출

In [32]:
product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11


## 54. 주문 날짜 변환과 주문 월 생성

In [33]:
completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

날짜 변환 실패: 0


In [34]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

In [36]:
completed_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match,product_name,category,product_match,order_month
0,1,1,100,3,102000,306000,123,2026-06-08,completed,both,도서 상품 100,도서,both,2026-06
1,2,1,87,5,25000,125000,123,2026-06-08,completed,both,도서 상품 087,도서,both,2026-06
2,3,1,7,3,142000,426000,123,2026-06-08,completed,both,도서 상품 007,도서,both,2026-06
3,4,1,9,3,193000,579000,123,2026-06-08,completed,both,스포츠 상품 009,스포츠,both,2026-06
4,13,6,83,3,24000,72000,87,2026-04-22,completed,both,전자기기 상품 083,전자기기,both,2026-04


## 55. 월별 매출

In [37]:
monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)
display(monthly_sales)

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-08,5869000,8,8,52
1,2025-09,15621000,18,18,127
2,2025-10,9955000,12,12,95
3,2025-11,25005000,26,26,246
4,2025-12,9808000,13,13,102
5,2026-01,9387000,12,12,89
6,2026-02,16981000,22,20,163
7,2026-03,11521000,18,16,113
8,2026-04,13135000,14,12,127
9,2026-05,17519000,22,21,179


## 56. 고객별 구매 금액

In [38]:
customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)

In [40]:
print(customer_sales.head())

   customer_id  total_sales  order_count  quantity_sold
0            3      3178000            2             26
1            4       603000            1              6
2            5      2004000            2             13
3            6      1349000            2             14
4            7      1173000            1              9


In [41]:
print(customer_sales.sort_values("total_sales", ascending=False).head(10))

#.sort_values("total_sales", ascending=False)
#display(product_sales.head(10))

    customer_id  total_sales  order_count  quantity_sold
76          117      4100000            5             48
62          102      3996000            4             35
51           83      3880000            4             39
21           30      3590000            5             32
29           40      3523000            4             27
13           20      3191000            2             25
0             3      3178000            2             26
70          111      3153000            3             38
42           66      3093000            4             30
97          147      2990000            2             21


## 57. 고객 속성 연결

개인정보 최소화를 위해 이름은 제외합니다.

In [42]:
customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()

In [43]:
customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

In [44]:
display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))

customer_match
both          100
left_only       0
right_only      0
Name: count, dtype: int64

,customer_id,total_sales,order_count,quantity_sold,gender,age,city,customer_match
76,117,4100000,5,48,F,65,성남,both
62,102,3996000,4,35,M,60,고양,both
51,83,3880000,4,39,F,22,수원,both
21,30,3590000,5,32,F,32,서울,both
29,40,3523000,4,27,M,23,서울,both
13,20,3191000,2,25,F,20,인천,both
0,3,3178000,2,26,F,61,성남,both
70,111,3153000,3,38,F,41,광주,both
42,66,3093000,4,30,F,39,서울,both
97,147,2990000,2,21,M,19,부산,both


## 58. 결과 폴더 생성

In [45]:
output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)

c:\dev\ai-data-analysis\reports\chapter04


In [46]:
outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}
for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

category_sales.csv True 309
product_sales.csv True 4833
monthly_sales.csv True 415
customer_sales.csv True 3448


## 60. 저장 결과 다시 읽기

In [47]:
saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
0,스포츠,31743000,85,67,295,100
1,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
3,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42


(7, 6)


## 61. 병합 점검 함수

In [48]:
def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )

위에 만든 함수 호출

In [49]:
check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)

[주문상세-주문]
병합 전 행 수: 764
병합 후 행 수: 764
order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64


## 62. 집계 합계 검증 함수

In [50]:
def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

In [51]:
check_total(
    name="카테고리별 매출",
    source_total=completed_items["line_total"].sum(),
    summary_total=category_sales["total_sales"].sum(),
)

[카테고리별 매출]
원본 합계: 148990000
요약 합계: 148990000
차이: 0


In [52]:
import pandas as pd

# ============================================================
# 0. 기본 구조 확인
# ============================================================

print("=== orders ===")
print(orders.info())
print()

print("=== order_items ===")
print(order_items.info())
print()

print("=== products ===")
print(products.info())
print()


# ============================================================
# 1. PK 및 데이터 상태 확인
# ============================================================

# orders.order_id는 주문별로 유일해야 함
print("orders.order_id 중복 수:",
      orders["order_id"].duplicated().sum())

# products.product_id는 상품별로 유일해야 함
print("products.product_id 중복 수:",
      products["product_id"].duplicated().sum())

# 분석에 필요한 주요 컬럼 결측치 확인
print("\n=== 주요 컬럼 결측치 ===")
print(
    orders[
        ["order_id", "customer_id", "order_status"]
    ].isna().sum()
)

print(
    order_items[
        ["order_id", "product_id", "quantity", "unit_price"]
    ].isna().sum()
)

print(
    products[
        ["product_id", "category"]
    ].isna().sum()
)

# completed 상태가 실제 존재하는지 확인
print("\n주문 상태:")
print(orders["order_status"].value_counts(dropna=False))


# ============================================================
# 2. completed 주문 필터링
# ============================================================

completed_orders = orders.loc[
    orders["order_status"] == "completed"
].copy()

print("\n완료 주문 수:", len(completed_orders))
print(
    "완료 주문 order_id 고유 개수:",
    completed_orders["order_id"].nunique()
)


# ============================================================
# 3. order_items + completed_orders 병합
# ============================================================

print("\n=== 첫 번째 merge ===")
print("병합 전 order_items 행 수:", len(order_items))
print("병합 전 completed_orders 행 수:", len(completed_orders))

merged_orders = order_items.merge(
    completed_orders[
        ["order_id", "customer_id"]
    ],
    on="order_id",
    how="inner",
    validate="many_to_one",
    indicator=True
)

print("병합 후 행 수:", len(merged_orders))
print("\nmerge 결과:")
print(merged_orders["_merge"].value_counts())

merged_orders = merged_orders.drop(columns="_merge")


# ============================================================
# 4. 미매칭 order_id 별도 확인
# ============================================================

order_check = order_items.merge(
    completed_orders[["order_id"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\n=== completed 주문과 매칭 여부 ===")
print(order_check["_merge"].value_counts())

# 주의:
# left_only에는 '미완료 주문'도 포함됩니다.
# 따라서 이것이 곧 잘못된 order_id라는 의미는 아닙니다.


# ============================================================
# 5. products 병합
# ============================================================

print("\n=== 두 번째 merge ===")
print("병합 전 행 수:", len(merged_orders))
print("products 행 수:", len(products))

merged = merged_orders.merge(
    products[
        ["product_id", "category"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("병합 후 행 수:", len(merged))
print("\n상품 merge 결과:")
print(merged["_merge"].value_counts())

# 매칭되지 않은 상품 확인
unmatched_products = merged.loc[
    merged["_merge"] == "left_only"
]

print("미매칭 상품 행 수:", len(unmatched_products))

merged = merged.drop(columns="_merge")


# ============================================================
# 6. line_total 계산
# ============================================================

merged["line_total"] = (
    merged["quantity"] * merged["unit_price"]
)


# ============================================================
# 7. 완료 주문 전체 합계 계산
# ============================================================

completed_total_sales = merged["line_total"].sum()

print("\n완료 주문 전체 매출:")
print(completed_total_sales)


# ============================================================
# 8. 카테고리별 집계
# ============================================================

category_summary = (
    merged
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum")
    )
    .sort_values(
        "total_sales",
        ascending=False
    )
)

print("\n=== 카테고리별 분석 결과 ===")
print(category_summary)


# ============================================================
# 9. 검증: 카테고리 합계 vs 완료 주문 전체 합계
# ============================================================

category_total_sales = category_summary["total_sales"].sum()

print("\n=== 매출 검증 ===")
print("완료 주문 전체 매출:", completed_total_sales)
print("카테고리별 매출 합계:", category_total_sales)
print(
    "차이:",
    completed_total_sales - category_total_sales
)

print(
    "합계 일치 여부:",
    category_total_sales == completed_total_sales
)

=== orders ===
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        300 non-null    int64
 1   customer_id     300 non-null    int64
 2   order_date      300 non-null    str  
 3   payment_method  300 non-null    str  
 4   order_status    300 non-null    str  
dtypes: int64(2), str(3)
memory usage: 11.8 KB
None

=== order_items ===
<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
dtypes: int64(5)
memory usage: 30.0 KB
None

=== products ===
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data colu

## 65. LLM 코드 검증표

| 검증 항목 | 확인 내용 | 결과 |
|---|---|---|
| DataFrame | 실제 변수명과 같은가? | 적합 - `orders`, `order_items`, `products` 사용 |
| 컬럼 | 실제 컬럼만 사용하는가? | 적합 - 실제 존재하는 컬럼을 사용하고, 분석 조건에 명시된 `line_total`만 파생변수로 생성 |
| 상태값 | `completed` 표기가 맞는가? | 적합 - `order_status == "completed"` 조건 사용 |
| 계산식 | `quantity × unit_price`인가? | 적합 - `line_total = quantity * unit_price`로 계산 |
| 분석 범위 | 완료 주문만 포함하는가? | 적합 - `completed_orders`를 생성하여 완료 주문만 분석 |
| 주문 수 | `nunique()`를 사용하는가? | 적합 - `order_id`의 고유 개수로 주문 수 계산 |
| 병합 키 | 실제 관계와 맞는가? | 적합 - `order_id`, `product_id`를 병합 키로 사용 |
| validate | `many_to_one`이 적용되었는가? | 적합 - 두 병합 모두 `validate="many_to_one"` 적용 |
| indicator | 미매칭을 확인하는가? | 적합 - `indicator=True`를 사용하여 `_merge` 값 확인 |
| 행 수 | 병합 전후를 비교하는가? | 적합 - 각 병합 전후 `len()`으로 행 수 확인 |
| 합계 | 원본과 요약 합계를 비교하는가? | 적합 - 완료 주문 전체 매출과 카테고리별 매출 합계를 비교 |
| 개인정보 | 원본 고객 정보를 요구하지 않는가? | 적합 - `customer_id`는 고유 고객 수 집계 목적으로만 사용하며 추가 개인정보를 요구하지 않음 |